# Inwiefern besteht ein Zusammenhang zwischen der Brutto CO2 Einspeisung, Trockenheit, Pflanzenmenge und der Bodentemperatur in Deutschland?

### bezogen auf die Hitzewelle 2018 in Deutschland

Was brauchen wir?

Erstmal: GPP Vergleich mit Trockenheit. Dafür: zwei Karten. Graph (Durschnittliche Trockenheit in Deutschland + Durschnittlicher GPP in Deutschland)

Plan::: Daten für GPP, Trockenheit, Bodentemperatur download. => auf gleiches Raster interpolieren. => alle drei auf nem Line Graph plotten

- CO2: CLMS_GPP_GLOBAL_300M_10DAILY_V2 2013-present
- Feuchtigkeit: CLMS_SSM_EUROPE_1KM_DAILY_V1 2014-present
- Temperatur: SENTINEL3_SLSTR_L2_LST 2016-present

=> 2016-present?

In [5]:
GPP_ID = "CLMS_GPP_GLOBAL_300M_10DAILY_V2"
SSM_ID = "CLMS_SSM_EUROPE_1KM_DAILY_V1"
LST_ID = "SENTINEL3_SLSTR_L2_LST"

In [6]:
spatial_extent = {
    'west': 5,
    'south': 46,
    'east': 15.5,
    'north': 56,
}

In [7]:
temporal_extent = ["2024-06-01", "2024-06-30"]

In [8]:
import folium
import geopandas as gpd
import json
import leafmap
import math
import matplotlib.pyplot as plt
import os
import openeo
import pyproj
import rasterio

from rasterio.plot import show
from shapely.geometry import Polygon

In [9]:
import openeo
from openeo.rest.auth.config import RefreshTokenStore

connection = openeo.connect("openeofed.dataspace.copernicus.eu")
connection.authenticate_oidc()
print(connection.describe_account())

Authenticated using refresh token.
{'info': {'oidc_userinfo': {'email': 'moritz.fechte@icloud.com', 'email_verified': True, 'family_name': 'Fe', 'given_name': 'Mo', 'name': 'Mo Fe', 'preferred_username': 'moritz.fechte@icloud.com', 'sub': '01b88e4b-35a5-4da2-8e10-edc6a6ce408f'}}, 'name': 'Mo Fe', 'user_id': '01b88e4b-35a5-4da2-8e10-edc6a6ce408f'}


### GPP Download

In [10]:
datacube = connection.load_collection(
    GPP_ID,
    spatial_extent=spatial_extent,
    temporal_extent = temporal_extent,
    bands=["gpp"],
)

datacube = datacube.max_time()
gpp = datacube.band('gpp')


Deutschland ausschneiden (germany.geojson Datei notwendig)

In [11]:
def mask_germany(datacube):
    germany = gpd.read_file("germany.geojson")
    geometry = germany.geometry.iloc[0].__geo_interface__

    return datacube.mask_polygon(
        geometry,
        srs="EPSG:4326"
    )

Zeit auf 10 Tage Intervall reduzieren

In [12]:
def reduce_to_decad(datacube):
    return datacube.aggregate_temporal_period(
        period="dekad",
        reducer="mean"
    )

Auflösung auf 1km reduzieren (resamplen)

In [ ]:
def downsample(datacube):
    return datacube.resample_spatial(
        resolution=1000,
        projection="EPSG:32632"
    )

In [14]:
gpp = mask_germany(gpp)
gpp = downsample(gpp)
#gpp = gpp.max_time()

gpp.download('output.tif')